# Exploring the Metadata of Data Frames

## What is metadata?

It is data about data.  Or as [Wikipedia says]( https://en.wikipedia.org/wiki/Metadata ):

> "data that provides information about other data", but not the content of the data itself, such as the text of a message or the image itself.

For example, given a file, the data are the contents of the file.  The metadata is the data about the file: the size, date, ownership, group membership, permissions, the file type, etc.

A sound track will contain audio, but the metadata would include the length of the recording, the recording date, the author's name, any copyright info, the performers names, etc.

An digital photo will contain the image with the metadata containing the date, time, exposure settings, GPS info, camera make/model, etc.  Usually this information is stored in the [EXIF portion]( https://en.wikipedia.org/wiki/Exif ) of the digital image.


For a data frame, the metadata is the number of rows and columns, the column names, the column data types, the row indices, the value counts of categorical data, the min/median/mean/max of continuous data, etc.

For a web resource ( e.g. file ), the metadata is what is in the headers of the response to an HTTP request, e.g. the server type, the last update date, the content-length, the content-type, etc.

Metadata is useful for understanding the data itself.  For ML algorithms it is important to understand the metadata to determine what types of ML algorithms are appropriate for the data and if any transformations need to happen if the data is not appropriate.


## Exploring dataframes


In [ ]:
%%capture
%%bash
apt-get update
apt-get install -y tree

In [ ]:
import pandas as pd


In [ ]:
url = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv"
housing = pd.read_csv( url )


Getting the shape ( rows, columns ) of a Data Frame


In [ ]:
housing.shape

In [ ]:
type(housing.shape)

In [ ]:
rows, columns = housing.shape
{
  "Rows": rows,
  "Columns" : columns
}

### Data types and nulls

#### Data from `.info()` ... or not

In [ ]:
housing.info()

Notice `.info()` provides five kinds of information:
- the number of rows and columns
- the number of rows of each column that are not-nulls
- the data type of the column
- summary counts of column data types
- the amount of memory used by the Data Frame

How can I extract that information to work with it?  For example, how can I use the data from `.info()` to list all columns that have nulls?

In [ ]:
type(housing.info())

In [ ]:
housing.info().transpose()

It's a "none" data type.  Hmmm.  `.info()` does not return any data.  It only displays it on the screen.

That means that you cannot do anything ( easily ) with the information that it is giving to you.

Let's put this aside for the moment.

### Some basic statistics


In [ ]:
housing.describe()

In [ ]:
type(housing.describe())

A data frame.  I can work with that using all the methods that a data frame has.

List the columns.

In [ ]:
housing.describe().columns

Count the number of columns.

In [ ]:
housing.describe().columns.size

Discover more columns.

In [ ]:
housing.describe( include = "all" ).columns.size

Use method chaining to customize the output.

In [ ]:
(
  housing
  .describe( include = "all" )
  .transpose()
  .astype({"count": int})
)

Notice that the names of the columns from the original data frame are the named index for the rows.

### Nulls

In [ ]:
housing.isnull()

In [ ]:
type(housing.isnull())

In [ ]:
housing.isnull().sum()

In [ ]:
type(housing.isnull().sum())

Again, notice that the names of the columns from the original data frame are the named index for the rows.

#### Calculate proportion of nulls

In [ ]:
nulls = housing.isnull().sum()
filter = nulls > 0
( nulls[ filter ].sort_values( ascending = False ) / housing.shape[0] * 100 ).round(1)

Again, notice that the names of the columns from the original data frame are the named index for the rows.

### Data types

In [ ]:
housing.dtypes

In [ ]:
type(housing.dtypes)

Again, notice that the names of the columns from the original data frame are the named index for the rows.

In [ ]:
housing.dtypes.value_counts()

### Memory usage

In [ ]:
housing.memory_usage(deep=False, index = False)

In [ ]:
housing.memory_usage(deep=False, index = False).sum()

In [ ]:
housing.memory_usage(deep=True, index = False).sum()

In [ ]:
type(housing.memory_usage(deep=True, index = False))

Again, notice that the names of the columns from the original data frame are the named index for the rows.

## Automate: first pass

Each of the above processes created either data frames or data series where the named index for the rows was the names of the columns from the original data frame.  That means that all these data sets can be concatenated together into a single data frame.


In [ ]:
# Assign current data frame to a generic name
pd.set_option('display.max_rows', 100)
df = housing

# Build a data frame to contain the metadata, using the generic name
df_info = pd.concat( [
  pd.DataFrame( [ df.isna().sum().to_dict() ], index = ["Nulls"] ).transpose().astype( { "Nulls": int } ),
  pd.DataFrame( [ df.dtypes.to_dict() ], index = [ "Data_Types"] ).transpose().astype( { "Data_Types": 'category' }),
  pd.DataFrame( [ df.memory_usage(deep=True, index = False).to_dict() ], index = ["Memory"] ).transpose(),
  df.describe( include = "all" ).transpose().astype( { "count": int } ).rename( columns = { "50%" : "median"} ),
], axis = "columns")
df_info


In [ ]:
df_info["IRQ"] = df_info["75%"] - df_info["25%"]
df_info["range"] = df_info["max"] - df_info["min"]
df_info["sum"] = df_info["mean"] * df_info["count"]
df_info.insert( 1, "Nulls_pct" , round( df_info["Nulls"] / df_info.shape[0], 2) )
df_info


In [ ]:
# Display some info from the meta data
print( f"Memory: { (df_info['Memory'].sum() / 1000 / 1000 ):_} MB" )
print( df_info["Data_Types"].value_counts().to_json() )
df_info.sort_values(by = ["Data_Types", "max"], ascending=[1,0])


Notes:
- remove columns with 20% or more of null values

- remove rows with 5% or fewer null values




In [ ]:
for dt in df_info["Data_Types"].unique():
  print(dt)

In [ ]:
df_info["Data_Types"].value_counts()


In [ ]:
for dt in df_info["Data_Types"].unique():
  filter = df_info["Data_Types"] == dt
  cols = df_info[ filter ].index
  housing[ cols ].info()

In [ ]:
df_info["Data_Types"]

In [ ]:
filter = df_info["Data_Types"].astype("str") == "object"
cols = df_info[ filter ].index
housing[ cols ].nunique().sort_values(ascending = False)


## Automate: second pass

### Blank data frame

In [ ]:
metatdata_df = pd.DataFrame()
metatdata_df

### Append data type info

In [ ]:
metatdata_df["Data_types"] = housing.dtypes
metatdata_df

### Append Null stats

In [ ]:
metatdata_df["Nulls"] = housing.isnull().sum()
metatdata_df

In [ ]:
metatdata_df["Nulls_pct"] = ( metatdata_df["Nulls"] / housing.shape[0] * 100 ).round(1)
metatdata_df

### Append memory usage

In [ ]:
metatdata_df["Memory"] = housing.memory_usage( deep = True)
metatdata_df

### Append unique counts and percentage


In [ ]:
metatdata_df["NUnique"] = housing.nunique()
metatdata_df


In [ ]:
metatdata_df["NUnique_pct"] = (housing.nunique()/len(housing)*100).round(1)
metatdata_df


### Append descriptive stats

Use .join() instead of .concatenate().  This ensures the named index is used to match values.

In [ ]:
metatdata_df = metatdata_df.join( housing.describe( include = "all" ).transpose() )
metatdata_df

### Adjust format of data frame

In [ ]:
metatdata_df = metatdata_df.astype( {"count": int }).rename( columns = {"50%": "median"})
metatdata_df

### Redo as a Function

In [ ]:
def metadata( dataframe ):
  '''Given a dataframe, returns a dataframe of metadata about the dataframe'''
  metadata_df = pd.DataFrame()
  metadata_df["Data_types"] = dataframe.dtypes
  metadata_df["Count"] = len(dataframe)
  metadata_df["Nulls"] = dataframe.isnull().sum()
  metadata_df["Nulls_pct"] = ( metadata_df["Nulls"] / metadata_df["Count"] * 100 ).round(1)
  metadata_df["Memory"] = dataframe.memory_usage( deep = True)
  metadata_df["NUnique"] = dataframe.nunique()
  metadata_df["NUnique_pct"] = (dataframe.nunique() / metadata_df["Count"] * 100).round(1)
  metadata_df = metadata_df.join( dataframe.describe( include = "all" ).transpose() )
  metadata_df["IRQ"] = metadata_df["75%"] - metadata_df["25%"]
  metadata_df["range"] = metadata_df["max"] - metadata_df["min"]
  metadata_df["sum"] = metadata_df["mean"] * metadata_df["count"]
  metadata_df = (
    metadata_df
    .astype( { "count" : int } )
    .rename( columns = {
      "25%" : "Q1_25%",
      "50%" : "Q2_median",
      "75%" : "Q3_75%",
      }
    )
  )
  return metadata_df


In [ ]:
md = metadata(housing)
md

A count of columns with each data type.

In [ ]:
md["Data_types"].value_counts()

All columns with more than some threshold of nulls.

In [ ]:
filter = ( md["Nulls_pct"] > 10 )
md["Nulls_pct"][ filter ].sort_values( ascending = False )

Identifier columns

In [ ]:
filter = ( md["NUnique_pct"] > 95 )
md[["count", "NUnique","NUnique_pct"]][ filter ].sort_values( by = ["NUnique_pct"], ascending = False )

Skew


In [ ]:
(md["mean"] > md["Q2_median"]).sum()

In [ ]:
(md["mean"] < md["Q2_median"]).sum()

## Automate: import


### Just one python file

In [ ]:
%%writefile metadata01.py
import pandas as pd

def metadata( dataframe ):
  '''Given a dataframe, returns a dataframe of metadata about the dataframe'''
  metadata_df = pd.DataFrame()
  metadata_df["Data_types"] = dataframe.dtypes
  metadata_df["Count"] = len(dataframe)
  metadata_df["Nulls"] = dataframe.isnull().sum()
  metadata_df["Nulls_pct"] = ( metadata_df["Nulls"] / metadata_df["Count"] * 100 ).round(1)
  metadata_df["Memory"] = dataframe.memory_usage( deep = True)
  metadata_df["NUnique"] = dataframe.nunique()
  metadata_df["NUnique_pct"] = (dataframe.nunique() / metadata_df["Count"] * 100).round(1)
  metadata_df = metadata_df.join( dataframe.describe( include = "all" ).transpose() )
  metadata_df = metadata_df.astype( { "count" : int } )
  if dataframe.select_dtypes(include=['number']).shape[1] :
    metadata_df["IRQ"] = metadata_df["75%"] - metadata_df["25%"]
    metadata_df["range"] = metadata_df["max"] - metadata_df["min"]
    metadata_df["sum"] = metadata_df["mean"] * metadata_df["count"]
    metadata_df = (
      metadata_df
      .rename( columns = {
        "25%" : "Q1_25%",
        "50%" : "Q2_median",
        "75%" : "Q3_75%",
        }
      )
    )
  return metadata_df

def cols_to_drop( dataframe ):
  '''Given a dataframe, returns columns that should likely be dropped'''
  md = metadata( dataframe )
  filter = md["Nulls_pct"] >= 40
  return md[ filter ]["Nulls_pct"].to_dict()

def likely_ids( dataframe ):
  '''Given a dataframe, returns a dictionary of likely ID columns'''
  md = metadata( dataframe )
  return md["NUnique_pct"][ md["NUnique_pct"] > 95 ].sort_values( ascending = False )


In [ ]:
!ls -la m*


In [ ]:
!cat -n metadata01.py


In [ ]:
import metadata01


In [ ]:
metadata01.metadata(df)

In [ ]:
import metadata01 as md01


In [ ]:
md01.metadata(df)

### GitHub



Upload your code to GitHub:
1. login to GitHub
1. create a repo, e.g. ds-toolbox
1. click Add file > Upload files
1. commit to the repository

Get the "raw" link:
1. click on the file
1. click Raw
1. copy the link in the address bar

( alternatively, right click on Raw and copy the link.  Use -L when using curl )



Download your toolkit file from GitHub.

In [ ]:
!ls -l


In [ ]:
# !curl -s -O https://raw.githubusercontent.com/rwcitek/ddc-example-repo/refs/heads/main/python_modules/metadata.py
# !curl -s -L -O https://github.com/rwcitek/ddc-example-repo/raw/refs/heads/main/python_modules/metadata.py

!curl -s -L -o metadata02.py https://github.com/rwcitek/ddc-example-repo/raw/refs/heads/main/python_modules/metadata.py


In [ ]:
!ls -l


In [ ]:
import metadata02


In [ ]:
md = metadata02.metadata(housing)
md


In [ ]:
metadata02.cols_to_drop(housing)


In [ ]:
metadata02?


In [ ]:
metadata02??


### Clone GitHub repo

How many packages are pre-installed in the CoLab environment?

In [ ]:
%%capture
pip = !pip3 list


In [ ]:
pip[:5], "...", pip[-5:]


In [ ]:
len(pip[2:])


In [ ]:
for i in pip:
  if "module" in i:
    print(i)


In [ ]:
!git clone https://github.com/rwcitek/ddc-example-repo.git


In [ ]:
ls -lF


In [ ]:
!tree ddc-example-repo


In [ ]:
cat -n ddc-example-repo/setup.py

In [ ]:
!pip install ./ddc-example-repo/


Now, how many packages are installed in the CoLab environment?

In [ ]:
pip = !pip3 list
len(pip[2:])


In [ ]:
for i in pip:
  if "module" in i:
    print(i)


In [ ]:
import python_modules as pm03


In [ ]:
pm03??

In [ ]:
cat -n /usr/local/lib/python3.12/dist-packages/python_modules/__init__.py


In [ ]:
pm03.metadata??


In [ ]:
ls -lF /usr/local/lib/python3.12/dist-packages/python_modules/

### Using pip and GitHub

In [ ]:
!pip install git+https://github.com/rwcitek/ddc-example-repo.git


In [ ]:
!pip list | grep module


In [ ]:
import python_modules as pm04


In [ ]:
pm04.hw()


In [ ]:
pm04.metadata(housing)


In [ ]:
pm04.foobar()